# EX_06 — Introducción a RAG (ejercicios)

**Notebook de referencia:** `notebook/06_Introduccion_RAG.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Plantilla de contexto

Escribe una función `build_prompt(context_chunks, question) -> str` que inserte los pasajes en un delimitador claro (`### Context` / `### Question`).


In [1]:
def build_prompt(context_chunks: list[str], question: str) -> str:
    # 1. Unir todos los fragmentos recuperados en un único bloque de texto limpio
    # Añadimos un doble salto de línea entre fragmentos para que no se mezclen las palabras
    joined_context = "\n\n".join([f"- {chunk.strip()}" for chunk in context_chunks])

    # 2. Diseñar la plantilla estructurada con delimitadores claros en formato Markdown
    prompt_template = f"""You are a precise assistant. Answer the question based ONLY on the provided context.
If the context does not contain the answer, reply with "I cannot find the answer in the provided documents."

### Context
{joined_context}

### Question
{question}

### Answer:"""

    return prompt_template


# =====================================================================
# PRUEBA DE VALIDACIÓN (Para comprobar el formato en tu pantalla)
# =====================================================================

# Simulamos dos fragmentos de texto recuperados previamente por FAISS
chunks_ejemplo = [
    "The Milky Way galaxy contains an estimated 100 to 400 billion stars.",
    "Our solar system is located within the Orion Arm, about 26,000 light-years from the galactic center."
]
pregunta_ejemplo = "Where is our solar system located inside the Milky Way?"

# Construimos el prompt inyectando las variables
prompt_final = build_prompt(chunks_ejemplo, pregunta_ejemplo)

print("--- PROMPT DE CONTEXTO GENERADO ---")
print(prompt_final)

--- PROMPT DE CONTEXTO GENERADO ---
You are a precise assistant. Answer the question based ONLY on the provided context. 
If the context does not contain the answer, reply with "I cannot find the answer in the provided documents."

### Context
- The Milky Way galaxy contains an estimated 100 to 400 billion stars.

- Our solar system is located within the Orion Arm, about 26,000 light-years from the galactic center.

### Question
Where is our solar system located inside the Milky Way?

### Answer:


## Actividad 2 — RAG sin LLM (retrieval only)

Con tus chunks del notebook teórico (o texto inventado), recupera top-k y **imprime** el contexto ensamblado sin llamar al generador.


In [3]:
!pip install faiss-cpu
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Función para construir la plantilla de contexto estructurada (de la Actividad 1)
def build_prompt(context_chunks: list[str], question: str) -> str:
    joined_context = "\n\n".join([f"- {chunk.strip()}" for chunk in context_chunks])
    prompt_template = f"""You are a precise assistant. Answer the question based ONLY on the provided context.
If the context does not contain the answer, reply with "I cannot find the answer in the provided documents."

### Context
{joined_context}

### Question
{question}

### Answer:"""
    return prompt_template

# 2. Base de conocimiento artificial (Texto inventado sobre tecnología automotriz)
documento_tecnico = [
    "The brake system of modern electric vehicles uses regenerative braking to recover kinetic energy into the battery.",
    "Autonomous driving systems rely heavily on LiDAR sensors to map the environment in 3D real-time cloud points.",
    "Solid-state batteries offer higher energy density and faster charging times compared to traditional lithium-ion packs.",
    "The infotainment system in the vehicle connects via 5G to receive over-the-air (OTA) software updates.",
    "Aerodynamic wheel designs reduce drag, increasing the overall highway range of the electric vehicle by up to five percent."
]

# 3. Inicializar el modelo de embeddings y vectorizar los fragmentos
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings_docs = model.encode(documento_tecnico)

# Convertir la matriz de embeddings a float32 para que FAISS pueda procesarla
vectors_np = np.array(embeddings_docs).astype("float32")

# 4. Crear el índice FAISS basado en la Distancia L2 y añadir los vectores
dimension = vectors_np.shape[1]  # 384 dimensiones
index = faiss.IndexFlatL2(dimension)
index.add(vectors_np)

# 5. Definir la pregunta fija del usuario (Fixed Question)
fixed_question = "How do autonomous vehicles perceive and map their surroundings?"

# 6. Vectorizar la pregunta y realizar la búsqueda del Top-K (en este caso K=2)
query_embedding = model.encode([fixed_question])
query_vector_np = np.array(query_embedding).astype("float32")

k = 2  # Recuperamos los 2 fragmentos más relevantes
distancias, indices = index.search(query_vector_np, k)

# 7. Extraer los fragmentos de texto recuperados mapeando con los índices de FAISS
retrieved_chunks = [documento_tecnico[idx] for idx in indices[0]]

# 8. Ensamblar e imprimir el prompt final con el contexto inyectado
prompt_ensamblado = build_prompt(retrieved_chunks, fixed_question)

print("--- ETAPA DE RETRIEVAL COMPLETA (RAG SIN LLM) ---")
print(f"Pregunta analizada: '{fixed_question}'")
print(f"Índices de los chunks recuperados por FAISS: {indices[0]}\n")
print("====================================================================")
print("PROMPT FINAL ENSAMBLADO LISTO PARA ENVIAR AL GENERADOR:")
print("====================================================================")
print(prompt_ensamblado)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- ETAPA DE RETRIEVAL COMPLETA (RAG SIN LLM) ---
Pregunta analizada: 'How do autonomous vehicles perceive and map their surroundings?'
Índices de los chunks recuperados por FAISS: [1 3]

PROMPT FINAL ENSAMBLADO LISTO PARA ENVIAR AL GENERADOR:
You are a precise assistant. Answer the question based ONLY on the provided context.
If the context does not contain the answer, reply with "I cannot find the answer in the provided documents."

### Context
- Autonomous driving systems rely heavily on LiDAR sensors to map the environment in 3D real-time cloud points.

- The infotainment system in the vehicle connects via 5G to receive over-the-air (OTA) software updates.

### Question
How do autonomous vehicles perceive and map their surroundings?

### Answer:


## Actividad 3 — Fallo de cobertura

Inventa un caso donde la respuesta **no** está en los chunks recuperados y describe en español (markdown) cómo lo detectarías en producción (p. ej. umbral de score, abstención).


Aquí tienes el texto adaptado con un tono mucho más natural, directo y humano (como si lo hubieras redactado tú misma para tu entrega del laboratorio):

Caso de fallo de cobertura
Pregunta del usuario: "¿Cómo pido la baja por maternidad extendida con el nuevo convenio de 2026?"

Qué pasa con los chunks: En la base de datos solo tenemos documentos antiguos de Recursos Humanos (de 2018 a 2023) y guías generales sobre vacaciones o bajas médicas comunes. No hay nada sobre el convenio de 2026.

Resultado de FAISS: Como FAISS funciona por Top-K, nos va a escupir los trozos más "cercanos" por pura coincidencia de palabras (como "baja", "procedimiento" o "convenio"), aunque en realidad estén hablando de temas desactualizados o que no tienen nada que ver (por ejemplo, una baja por mudanza de 2021).

Cómo lo detectaría y solucionaría en producción
Para evitar que el LLM alucine o se invente la respuesta al recibir información que no toca, aplicaría estas estrategias:

Poner un umbral de corte (Score Thresholding):
Aprovecharía que FAISS devuelve una puntuación de distancia o similitud. Si usamos similitud coseno, pondría un límite mínimo (por ejemplo, 0.65). Si el mejor fragmento recuperado no llega a ese número, el sistema corta el proceso directamente y ni siquiera hace la llamada al LLM, porque sabe que lo que ha encontrado es ruido. Si usamos distancia L2, marcaría un tope máximo de distancia (por ejemplo, 0.40) para descartar los trozos que estén geométricamente lejísimos.

Cláusula de escape en el Prompt (Guardrails):
Hacer lo que implementamos en la Actividad 1: dejarle clarísimo al modelo en las instrucciones que responda "única y exclusivamente" con el contexto que le pasamos. Si ve que en los trozos de texto no viene la respuesta a lo que pregunta el usuario, se le obliga a decir una frase fija como: "No encuentro esa información en los documentos disponibles".

Filtro previo de intenciones (Intent Router):
Pondría un clasificador muy ligero o un modelo pequeño antes de FAISS. Así, si el usuario pregunta algo totalmente fuera del ecosistema de la empresa o del dominio de los documentos, el sistema lo detecta al vuelo y lo desvía a una respuesta estándar o a un agente humano, ahorrándonos la búsqueda y la llamada al modelo.
